[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jorgeanais/libro_aprendizaje_profundo/blob/main/cap1/100_red_estilo_pytorch.ipynb)

---
title: Construyendo una red neuronal artificial al estilo PyTorch
subject: Aprendizaje Profundo
subtitle: 
short_title: RN al estilo PyTorch
authors:
  - name: Jorge Anais
    orcid: 0000-0001-9051-1338
    email: jrganais@gmail.com
license: MIT
---

**Objetivo**: Programar una red neuronal con el framework de diferenciación automática PyTorch.

## Importación librerías

In [14]:
import sys
import time

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader


# Fijamos una semilla para que los experimentos sean reproducibles
t_cg = torch.manual_seed(2202)

## Introducción


En esta etapa, aumentaremos el nivel de abstracción respecto a la actividad anterior. Para ello, aprovecharemos que PyTorch ya incluye diversas capas predefinidas; por ejemplo, las redes fully connected se encuentran implementadas en `torch.nn.Linear`. Asimismo, la librería ofrece múltiples funciones de activación, como la sigmoide y la tangente hiperbólica, entre otras.

Al utilizar este enfoque, PyTorch gestiona los parámetros automáticamente siguiendo estándares de la industria. Por ejemplo, en lugar de una asignación aleatoria simple, emplea heurísticas de inicialización optimizadas según la literatura técnica. Aunque profundizaremos en estas estrategias la próxima semana, por ahora nos enfocaremos en dominar este mayor nivel de abstracción

## Definiendo la arquitectura

In [15]:
# Red estilo pytorch
class FFNN(torch.nn.Module):
    def __init__(self, d0: int = 784, d1: int = 8, d2: int = 4):
        super(FFNN, self).__init__()

        # Definimos capas (automáticamente se registran como parametros)
        self.fc1 = torch.nn.Linear(d0, d1, bias=True)
        self.fc2 = torch.nn.Linear(d1, d2, bias=True)
        self.fc3 = torch.nn.Linear(d2,  1, bias=True)

    # Computa la pasada hacia adelante
    def forward(self, x: torch.Tensor):

        u1 = self.fc1(x)
        h1 = torch.tanh(u1)
        u2 = self.fc2(h1)
        h2 = torch.sigmoid(u2)
        u3 = self.fc3(h2)
        y_pred = torch.sigmoid(u3)

        return y_pred

## Dataset

Igual a como lo definimos anteriormente.

In [22]:
class CustomDataSet(Dataset):
    def __init__(self, csv_path: Path):
        """Lee el archivo CSV con los datos y genera un Dataset"""
        df = pd.read_csv(csv_path)

        labels = torch.tensor(df["label"].values, dtype=torch.long)
        self.labels = labels.unsqueeze(1)  # shape: (N, 1)


        pixel_cols = [c for c in df.columns if c.startswith("pixel_")]
        pixels = df[pixel_cols].values.astype(np.float32) / 255.0  # Normalización
        self.flatten_images = torch.tensor(pixels)  # shape: (N, 784)

        self.num_features = len(pixel_cols)

    # Debemos definir __len__ para retornar el tamaño del dataset
    def __len__(self):
        return len(self.labels)

    # Debemos definir __getitem__ para retornar el i-ésimo ejemplo en nuestro dataset.
    def __getitem__(self, idx):
        flatten_image = self.flatten_images[idx]  # shape: (784,)
        label = self.labels[idx].float()

        return flatten_image, label

## Bucle de entrenamiento

In [17]:
def loop_FFNN_pytorch_style(
    dataset: Dataset,
    batch_size: int,   # tamaño del lote
    d1: int,           # número de neuronas en la capa 1
    d2: int,           # número de neuronas en la capa 2
    lr: float,         # tasa de aprendizaje
    epochs: int,
    run_in_GPU: bool=True,
    reports_every: int=1,
):

    # Define un tipo para los tensores según si correrá en la GPU o no
    device = 'cuda' if run_in_GPU else 'cpu'

    # d0 es la cantidad de características (features)
    d0 = dataset.num_features

    # Cantidad de ejemplos
    N = len(dataset)

    # Crea la red
    red = FFNN(d0, d1, d2)

    # Pasa la red al dispositivo elegido
    red.to(device)

    # Muestra la cantidad de parámetros
    print('Número de parámetros de la red:', red)

    # Crea un dataloader desde el dataset
    data = DataLoader(dataset, batch_size, shuffle=True)

    # Descenso de gradiente
    optimizador = torch.optim.SGD(red.parameters(), lr)

    # Define una perdida
    perdida = torch.nn.BCELoss()

    # Comienza el entrenamiento
    tiempo_epochs = 0
    for e in range(1,epochs+1):
        inicio_epoch = time.process_time()

        for (x,y) in data:
            # Asegura de pasarlos a la GPU si fuera necesario
            x, y = x.to(device), y.to(device)

            # Computa la pasada hacia adelante (forward)
            y_pred = red.forward(x)

            # Computa la función de pérdida
            L = perdida(y_pred,y)

            # Computa los gradientes hacia atrás (backpropagation)
            L.backward()

            # Descenso de gradiente para actualizar los parámetros
            optimizador.step()

            # Limpia los gradientes
            optimizador.zero_grad()

        tiempo_epochs += time.process_time() - inicio_epoch

        if e % reports_every == 0:
            # Calcula la certeza de las predicciones sobre todo el conjunto
            X = dataset.flatten_images.to(device)
            Y = dataset.labels.to(device)

            # Predice usando la red
            Y_PRED = red.forward(X)

            # Calcula la pérdida de todo el conjunto
            L_total = perdida(Y_PRED, Y)

            # Elige una clase dependiendo del valor de Y_PRED
            Y_PRED_BIN = (Y_PRED >= 0.5).float()

            correctos = torch.sum(Y_PRED_BIN == Y).item()
            acc = (correctos / N) * 100

            sys.stdout.write(
                f"Epoch:{e:03d} Acc:{acc:.2f} Loss:{L_total:.4f} Tiempo/epoch:{tiempo_epochs/e:.3f}s\n"
            )

### <font color="teal">Actividad 1</font>

<font color="teal">Compara el bucle de entrenamiento de arriba con el que realizamos en la actividad anterior. ¿Qué diferencias encuentras? </font>

## Entrenando la red

In [18]:
csv_path = Path("/content/mnist_digits_0_1.csv")
dataset = CustomDataSet(csv_path)

In [23]:
loop_FFNN_pytorch_style(
    dataset=dataset,
    batch_size=8,   # tamaño del lote
    d1=8,           # número de neuronas en la capa 1
    d2=4,           # número de neuronas en la capa 2
    lr=0.001,       # tasa de aprendizaje
    epochs=10,      # numero de épocas
    run_in_GPU=False,
    reports_every=1,
)

Número de parámetros de la red: FFNN(
  (fc1): Linear(in_features=784, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=1, bias=True)
)
Epoch:001 Acc:53.29 Loss:0.6733 Tiempo/epoch:1.803s
Epoch:002 Acc:53.29 Loss:0.6464 Tiempo/epoch:1.655s
Epoch:003 Acc:92.92 Loss:0.5983 Tiempo/epoch:1.618s
Epoch:004 Acc:99.21 Loss:0.5235 Tiempo/epoch:1.591s
Epoch:005 Acc:99.57 Loss:0.4315 Tiempo/epoch:1.578s
Epoch:006 Acc:99.68 Loss:0.3409 Tiempo/epoch:1.569s
Epoch:007 Acc:99.70 Loss:0.2660 Tiempo/epoch:1.579s
Epoch:008 Acc:99.71 Loss:0.2105 Tiempo/epoch:1.642s
Epoch:009 Acc:99.74 Loss:0.1706 Tiempo/epoch:1.645s
Epoch:010 Acc:99.76 Loss:0.1419 Tiempo/epoch:1.633s
